In [ ]:
# ===================== 1️⃣ تثبيت المكتبات والمستودعات =====================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# تثبيت جميع المكتبات المطلوبة
!pip install torch torchvision pillow opencv-python fastapi uvicorn nest_asyncio pyngrok mediapipe==0.10.21
!pip install basicsr facelib gfpgan scikit-image numpy timm peft
!pip install lpips git+https://github.com/openai/CLIP.git ftfy regex tqdm
!pip install huggingface-hub==0.20.3 diffusers==0.26.3 transformers==4.36.2 accelerate==0.26.1 safetensors
!pip install peft==0.8.2 accelerate==0.25.0
!pip install xformers --upgrade



  Using cached basicsr-1.4.2.tar.gz (172 kB)
  Preparing metadata (setup.py) ... done
ERROR: Ignored the following versions that require a different python version: 1.0 Requires-Python >=3.5, <3.8; 1.1 Requires-Python >=3.5, <3.8; 1.2 Requires-Python >=3.5, <3.8; 1.2.1 Requires-Python >=3.5, <3.8
ERROR: Could not find a version that satisfies the requirement facelib (from versions: none)
ERROR: No matching distribution found for facelib
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-mm0tcd9u
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-mm0tcd9u
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.1/330.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 48.8 MB/s 

In [ ]:

# استنساخ المستودعات المطلوبة
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix
!git clone https://github.com/sczhou/CodeFormer
%cd CodeFormer
!python basicsr/setup.py develop
%cd /content


Cloning into 'pytorch-CycleGAN-and-pix2pix'...
remote: Enumerating objects: 2619, done.
remote: Total 2619 (delta 0), reused 0 (delta 0), pack-reused 2619 (from 1)
Receiving objects: 100% (2619/2619), 8.23 MiB | 14.44 MiB/s, done.
Resolving deltas: 100% (1654/1654), done.
Cloning into 'CodeFormer'...
remote: Enumerating objects: 620, done.
remote: Counting objects: 100% (303/303), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 620 (delta 212), reused 185 (delta 185), pack-reused 317 (from 2)
Receiving objects: 100% (620/620), 17.32 MiB | 15.22 MiB/s, done.
Resolving deltas: 100% (300/300), done.
/content/CodeFormer
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip i

In [ ]:
# ===================== 2️⃣ استيراد المكتبات =====================
import os
import sys
import time
import re
import glob
import shutil
import tempfile
import asyncio
import signal
import subprocess
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import cv2
from PIL import Image
import mediapipe as mp
import torch
import torch.nn as nn
from torchvision import transforms
import torch.nn.functional as F
import xformers

# استيراد نماذج التقييم
import lpips
import clip

# استيراد FastAPI
from fastapi import FastAPI, File, UploadFile, Form, HTTPException, Request
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates
import nest_asyncio
import uvicorn
from pyngrok import ngrok

# استيراد Diffusers بالنسخ المتوافقة
from diffusers import StableDiffusionInpaintPipeline, DPMSolverMultistepScheduler
from diffusers.utils import load_image
from diffusers.loaders import AttnProcsLayers
from peft import LoraConfig, get_peft_model, PeftModel

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytre

In [ ]:
# ===================== 3️⃣ إعداد متغيرات البيئة =====================
os.environ["DIFFUSERS_IGNORE_UNUSED_PIPELINES"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # تقليل logs TensorFlow

# ===================== 4️⃣ تهيئة التطبيق العام =====================
nest_asyncio.apply()
app = FastAPI(title="Teeth Enhancement Models")

# إعداد مجلدات القوالب والملفات الثابتة
os.makedirs("static", exist_ok=True)
app.mount("/static", StaticFiles(directory="static"), name="static")
templates = Jinja2Templates(directory=".")

# إعداد ngrok
NGROK_AUTH_TOKEN = "3AAe9xiXJemH6HjZPnDpHklRbw7_5sGjDTYeNqNVDWrscCrcg"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# ===================== 5️⃣ دوال التحقق من البرومبت =====================
def is_valid_prompt(prompt: str) -> bool:
    """
    التحقق من صحة البرومبت - يسمح فقط بالأحرف العربية والإنجليزية والرموز الأساسية
    بدون رموز متتالية أو في البداية/النهاية
    """
    if not prompt.strip():
        return True

    # تحقق من وجود رموز متتالية
    if re.search(r'[,._\-!?،]{2,}', prompt):
        return False

    # تحقق من أن الرموز ليست في البداية أو النهاية
    if prompt.startswith((' ', ',', '.', '_', '-', '!', '?', '،')) or \
       prompt.endswith((' ', ',', '.', '_', '-', '!', '?', '،')):
        return False

    # تحقق من أن النص يحتوي فقط على الأحرف المسموحة
    if not re.match(r'^[a-zA-Z\u0600-\u06FF\s,._\-!?،]*$', prompt):
        return False

    return True

def sanitize_prompt(prompt: str) -> str:
    """
    تنظيف البرومبت وإزالة الرموز غير المسموحة
    """
    if not prompt:
        return ""

    # السماح فقط بالأحرف العربية والإنجليزية والرموز الأساسية
    cleaned = re.sub(r'[^a-zA-Z\u0600-\u06FF\s,._\-!?،]', '', prompt)

    # إزالة الرموز المتتالية
    cleaned = re.sub(r'([,._\-!?،])\1+', r'\1', cleaned)

    # إزالة المسافات الزائدة
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    # إزالة الرموز من البداية والنهاية
    cleaned = re.sub(r'^[,._\-!?،\s]+', '', cleaned)
    cleaned = re.sub(r'[,._\-!?،\s]+$', '', cleaned)

    return cleaned

# ===================== 6️⃣ تهيئة نماذج التقييم =====================
def load_evaluation_models():
    print("⏳ جارٍ تحميل نماذج التقييم (LPIPS و CLIP)...")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # تحميل نموذج LPIPS
    lpips_model = lpips.LPIPS(net='vgg').to(device)

    # تحميل نموذج CLIP
    clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

    print("✅ تم تحميل نماذج التقييم بنجاح")
    return lpips_model, clip_model, clip_preprocess, device

# ===================== 7️⃣ دوال MediaPipe =====================
mp_face_mesh = mp.solutions.face_mesh

def detect_landmarks_mediapipe(image):
    """اكتشاف نقاط الوجه باستخدام MediaPipe"""
    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.3) as face_mesh:

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image_rgb)
        if not results.multi_face_landmarks:
            raise Exception("لم يتم العثور على أي وجه")
        face_landmarks = results.multi_face_landmarks[0]
        h, w = image.shape[:2]
        landmarks_points = [(int(l.x * w), int(l.y * h)) for l in face_landmarks.landmark]
        return landmarks_points

def get_mouth_indices_mediapipe():
    """مؤشرات نقاط الفم"""
    return [13, 14, 312, 311, 310, 415, 308, 324, 318, 402, 317,
            14, 87, 178, 88, 95, 78, 191, 80, 81, 82, 61, 146, 91, 181]

def ensure_roi_in_bounds(image, x, y, w, h):
    """التأكد من أن المنطقة المحددة داخل الصورة"""
    height, width = image.shape[:2]
    x = max(0, x)
    y = max(0, y)
    w = min(w, width - x)
    h = min(h, height - y)
    if w <= 0 or h <= 0:
        raise ValueError("منطقة الفم خارج حدود الصورة")
    return x, y, w, h

# ===================== 8️⃣ دوال قص الوجه =====================
def segment_face_only(image_pil, face_mesh, margin=20, output_size=512):
    try:
        image_bgr = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
        height, width, _ = image_bgr.shape
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image_rgb)

        if not results.multi_face_landmarks:
            return None

        face_landmarks = results.multi_face_landmarks[0]
        face_oval_indices = mp.solutions.face_mesh.FACEMESH_FACE_OVAL

        points = []
        for index_tuple in face_oval_indices:
            index = index_tuple[0]
            landmark = face_landmarks.landmark[index]
            x = int(landmark.x * width)
            y = int(landmark.y * height)
            points.append([x, y])

        points = np.array(points)
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)

        x_min = max(x_min - margin, 0)
        y_min = max(y_min - margin, 0)
        x_max = min(x_max + margin, width - 1)
        y_max = min(y_max + margin, height - 1)

        face_crop = image_bgr[y_min:y_max, x_min:x_max]
        mask_crop = np.zeros_like(face_crop[:, :, 0])
        shifted_points = points - [x_min, y_min]
        convex_hull = cv2.convexHull(shifted_points)
        cv2.fillConvexPoly(mask_crop, convex_hull, 255)

        final_crop = np.ones_like(face_crop, dtype=np.uint8) * 255
        final_crop[mask_crop == 255] = face_crop[mask_crop == 255]

        final_resized = cv2.resize(final_crop, (output_size, output_size), interpolation=cv2.INTER_AREA)
        final_pil = Image.fromarray(cv2.cvtColor(final_resized, cv2.COLOR_BGR2RGB))
        return final_pil

    except Exception as e:
        print("✗ خطأ بقص الوجه:", e)
        return None

# ===================== 9️⃣ دوال استخراج واستبدال الفم =====================
def remove_old_mouth(target_face, landmarks):
    """إزالة الفم القديم"""
    cleaned_face = target_face.copy()
    mouth_indices = get_mouth_indices_mediapipe()
    mouth_points = [landmarks[i] for i in mouth_indices if i < len(landmarks)]
    if len(mouth_points) < 3:
        raise ValueError("مافي نقاط فم كافية")
    hull = cv2.convexHull(np.array(mouth_points, dtype=np.int32))
    mask = np.zeros(target_face.shape[:2], dtype=np.uint8)
    cv2.fillConvexPoly(mask, hull, 255)
    cleaned_face = cv2.inpaint(cleaned_face, mask, 2, cv2.INPAINT_TELEA)
    return cleaned_face, mouth_points

def extract_mouth_precise(source_image, landmarks):
    """استخراج الفم بدقة"""
    mouth_indices = get_mouth_indices_mediapipe()
    mouth_points = [landmarks[i] for i in mouth_indices if i < len(landmarks)]
    if len(mouth_points) < 3:
        raise ValueError("مافي نقاط فم كافية")
    hull = cv2.convexHull(np.array(mouth_points, dtype=np.int32))
    mask = np.zeros(source_image.shape[:2], dtype=np.uint8)
    cv2.fillConvexPoly(mask, hull, 255)
    x, y, w, h = cv2.boundingRect(hull)
    x, y, w, h = ensure_roi_in_bounds(source_image, x, y, w, h)
    mouth_cropped = source_image[y:y+h, x:x+w].copy()
    mask_cropped = mask[y:y+h, x:x+w].copy()
    return mouth_cropped, mask_cropped, (x, y, w, h), mouth_points

def align_mouth(source_mouth, target_rect, target_shape):
    """محاذاة الفم"""
    x, y, w, h = target_rect
    x, y, w, h = ensure_roi_in_bounds(np.zeros(target_shape, dtype=np.uint8), x, y, w, h)
    return cv2.resize(source_mouth, (w, h)) if w > 0 and h > 0 else None

def adjust_colors_light(source_mouth, target_region):
    """ضبط الألوان والإضاءة"""
    if target_region.size == 0 or source_mouth.size == 0:
        return source_mouth
    source_lab = cv2.cvtColor(source_mouth, cv2.COLOR_BGR2LAB).astype(np.float32)
    target_lab = cv2.cvtColor(target_region, cv2.COLOR_BGR2LAB).astype(np.float32)
    src_mean_L, trg_mean_L = np.mean(source_lab[:, :, 0]), np.mean(target_lab[:, :, 0])
    source_lab[:, :, 0] = np.clip(source_lab[:, :, 0] + (trg_mean_L - src_mean_L) * 0.3, 0, 255)
    return cv2.cvtColor(source_lab.astype(np.uint8), cv2.COLOR_LAB2BGR)

def blend_mouth_natural(target_face, mouth_region, mouth_mask, target_rect):
    """دمج الفم بشكل طبيعي"""
    x, y, w, h = target_rect
    x, y, w, h = ensure_roi_in_bounds(target_face, x, y, w, h)
    if w <= 0 or h <= 0:
        raise ValueError("منطقة الفم غير صالحة")

    mouth_mask = cv2.resize(mouth_mask, (w, h))
    target_region = target_face[y:y+h, x:x+w]
    mouth_adjusted = adjust_colors_light(mouth_region, target_region)

    mask_float = (mouth_mask.astype(np.float32) / 255.0)
    result = target_face.copy()
    region_result = result[y:y+h, x:x+w].astype(np.float32)

    for c in range(3):
        region_result[:, :, c] = region_result[:, :, c] * (1 - mask_float) + mouth_adjusted[:, :, c] * mask_float

    result[y:y+h, x:x+w] = region_result.astype(np.uint8)
    return result

def replace_mouth_natural(source_path, target_path):
    """استبدال الفم بشكل طبيعي (لنموذج Pix2Pix)"""
    source = cv2.imread(source_path)
    target = cv2.imread(target_path)

    source_landmarks = detect_landmarks_mediapipe(source)
    target_landmarks = detect_landmarks_mediapipe(target)

    target_no_mouth, target_mouth_points = remove_old_mouth(target, target_landmarks)
    src_mouth, src_mask, src_rect, _ = extract_mouth_precise(source, source_landmarks)

    target_x, target_y, target_w, target_h = cv2.boundingRect(np.array(target_mouth_points))
    target_x, target_y, target_w, target_h = ensure_roi_in_bounds(target, target_x, target_y, target_w, target_h)

    aligned_mouth = align_mouth(src_mouth, (target_x, target_y, target_w, target_h), target.shape)
    aligned_mask = cv2.resize(src_mask, (target_w, target_h))

    return blend_mouth_natural(target_no_mouth, aligned_mouth, aligned_mask, (target_x, target_y, target_w, target_h))

# ===================== 🔟 دوال CodeFormer والتحسين =====================
def run_codeformer_restore(input_path, output_path, w=0.7):
    """تشغيل CodeFormer للتحسين"""
    try:
        script_path = "/content/CodeFormer/inference_codeformer.py"

        if not os.path.exists(script_path):
            print("❌ CodeFormer غير مثبت، استخدام الصورة الأصلية")
            shutil.copy(input_path, output_path)
            return 0, "CodeFormer not installed"

        if not os.path.exists(input_path):
            raise FileNotFoundError(f"ملف الإدخال غير موجود: {input_path}")

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        temp_output_dir = tempfile.mkdtemp()
        temp_output_path = os.path.join(temp_output_dir, "codeformer_output")

        cmd = [
            'python', script_path,
            '-i', input_path,
            '-o', temp_output_dir,
            '-w', str(w),
            '--face_upsample',
            '--bg_upsampler', 'none'
        ]

        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                             text=True, timeout=300)

        result_files = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
            result_files.extend(glob.glob(os.path.join(temp_output_dir, '**', ext), recursive=True))
            result_files.extend(glob.glob(os.path.join(temp_output_dir, ext)))

        if result_files:
            latest_file = max(result_files, key=os.path.getctime)
            shutil.copy(latest_file, output_path)
            print(f"✅ تم العثور على ملف الإخراج: {latest_file}")
        else:
            print("❌ CodeFormer لم ينشئ أي ملفات، استخدام الصورة الأصلية")
            shutil.copy(input_path, output_path)

        shutil.rmtree(temp_output_dir, ignore_errors=True)
        return res.returncode, res.stdout + "\n" + res.stderr

    except subprocess.TimeoutExpired:
        print("⏰ انتهى وقت CodeFormer، استخدام الصورة الأصلية")
        shutil.copy(input_path, output_path)
        return 0, "Timeout"
    except Exception as e:
        print(f"❌ خطأ في CodeFormer: {e}")
        shutil.copy(input_path, output_path)
        return 0, f"Error: {str(e)}"

def enhance_image_alternative(input_path, output_path):
    """تحسين بديل باستخدام CLAHE"""
    try:
        img = cv2.imread(input_path)
        if img is None:
            shutil.copy(input_path, output_path)
            return False

        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, b))
        enhanced = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

        kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
        sharpened = cv2.filter2D(enhanced, -1, kernel)

        cv2.imwrite(output_path, sharpened, [cv2.IMWRITE_JPEG_QUALITY, 95])
        return True

    except Exception as e:
        print(f"❌ خطأ في التحسين البديل: {e}")
        shutil.copy(input_path, output_path)
        return False

# ===================== 1️⃣1️⃣ دوال تقييم الجودة =====================
def segment_face_only_eval(image_path, face_mesh, margin=20, output_size=512):
    """قص الوجه للتقييم"""
    try:
        image_bgr = cv2.imread(image_path)
        if image_bgr is None:
            return None

        height, width, _ = image_bgr.shape
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image_rgb)

        if not results.multi_face_landmarks:
            return None

        face_landmarks = results.multi_face_landmarks[0]
        points = []
        for index_tuple in mp.solutions.face_mesh.FACEMESH_FACE_OVAL:
            index = index_tuple[0]
            landmark = face_landmarks.landmark[index]
            x = int(landmark.x * width)
            y = int(landmark.y * height)
            points.append([x, y])

        points = np.array(points)
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)

        x_min = max(x_min - margin, 0)
        y_min = max(y_min - margin, 0)
        x_max = min(x_max + margin, width - 1)
        y_max = min(y_max + margin, height - 1)

        face_crop = image_bgr[y_min:y_max, x_min:x_max]
        mask_crop = np.zeros_like(face_crop[:, :, 0])
        shifted_points = points - [x_min, y_min]
        convex_hull = cv2.convexHull(shifted_points)
        cv2.fillConvexPoly(mask_crop, convex_hull, 255)

        final_crop = np.ones_like(face_crop, dtype=np.uint8) * 255
        final_crop[mask_crop == 255] = face_crop[mask_crop == 255]

        final_resized = cv2.resize(final_crop, (output_size, output_size), interpolation=cv2.INTER_AREA)
        return final_resized

    except:
        return None

def load_lpips_image(img, device):
    """تحميل صورة لـ LPIPS"""
    transform = transforms.ToTensor()
    tensor = transform(img).unsqueeze(0).to(device) * 2 - 1
    return tensor

def load_clip_image(img, clip_preprocess, device):
    """تحميل صورة لـ CLIP"""
    return clip_preprocess(img).unsqueeze(0).to(device)

def compute_similarity(img1_path, img2_path, lpips_model, clip_model, clip_preprocess, device):
    """حساب التشابه بين صورتين"""
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5)

    face1 = segment_face_only_eval(img1_path, face_mesh)
    face2 = segment_face_only_eval(img2_path, face_mesh)
    face_mesh.close()

    if face1 is None or face2 is None:
        return None, None, None

    face1_pil = Image.fromarray(cv2.cvtColor(face1, cv2.COLOR_BGR2RGB))
    face2_pil = Image.fromarray(cv2.cvtColor(face2, cv2.COLOR_BGR2RGB))

    # LPIPS
    img1_lpips = load_lpips_image(face1_pil, device)
    img2_lpips = load_lpips_image(face2_pil, device)

    with torch.no_grad():
        lpips_dist = lpips_model(img1_lpips, img2_lpips).item()
        lpips_sim = 1 - lpips_dist

    # CLIP
    img1_clip = load_clip_image(face1_pil, clip_preprocess, device)
    img2_clip = load_clip_image(face2_pil, clip_preprocess, device)
    with torch.no_grad():
        f1 = clip_model.encode_image(img1_clip).float()
        f2 = clip_model.encode_image(img2_clip).float()
        f1 /= f1.norm(dim=-1, keepdim=True)
        f2 /= f2.norm(dim=-1, keepdim=True)
        clip_sim = (f1 @ f2.T).item()

    final_similarity = (lpips_sim + clip_sim) / 2
    return lpips_sim * 100, clip_sim * 100, final_similarity * 100

# ===================== 1️⃣2️⃣ تهيئة نموذج Pix2Pix =====================
def load_pix2pix_model():
    """تحميل نموذج Pix2Pix"""
    print("⏳ جارٍ تحميل نموذج Pix2Pix...")
    MODEL_PATH = '/content/drive/MyDrive/final_pix2pix_models/teeth_pix2pix_final/496_net_G.pth'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    sys.path.append('/content/pytorch-CycleGAN-and-pix2pix/models')
    from networks import UnetGenerator

    model = UnetGenerator(input_nc=3, output_nc=3, num_downs=8)
    state = torch.load(MODEL_PATH, map_location=device)
    if 'state_dict' in state:
        model.load_state_dict(state['state_dict'])
    else:
        model.load_state_dict(state)
    model.to(device)
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    return model, transform, device

# ===================== 1️⃣3️⃣ تهيئة نموذج Stable Diffusion Inpainting مع LoRA =====================
def load_sd_inpainting_model():
    """
    تحميل نموذج Stable Diffusion Inpainting مع دعم LoRA
    متوافق مع الإصدارات: diffusers==0.26.3, transformers==4.36.2, peft==0.8.2
    """
    print("=" * 60)
    print("🔧 جارٍ تحميل نموذج Stable Diffusion Inpainting...")
    print("=" * 60)

    MODEL_NAME = "runwayml/stable-diffusion-inpainting"
    LORA_BASE_PATH = '/content/drive/MyDrive/inpaint_lora_output_ffinal'

    # البحث عن أفضل مجلد LoRA
    possible_lora_paths = [
        '/content/drive/MyDrive/inpaint_lora_output_ffinal/checkpoint-10200',

    ]

    lora_path = None
    for path_pattern in possible_lora_paths:
        if '*' in path_pattern:
            matches = glob.glob(path_pattern)
            if matches:
                lora_path = matches[0]
                break
        elif os.path.exists(path_pattern):
            lora_path = path_pattern
            break

    if lora_path:
        print(f"📁 تم العثور على مسار LoRA: {lora_path}")
        print("📄 محتويات المجلد:")
        for file in os.listdir(lora_path):
            if file.endswith(('.safetensors', '.bin', '.json')):
                print(f"   - {file}")
    else:
        print("⚠️ لم يتم العثور على مجلد LoRA")
        lora_path = None

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"💻 استخدام الجهاز: {device}")

    try:
        # 1️⃣ تحميل النموذج الأساسي
        print("\n🔧 1. تحميل النموذج الأساسي...")
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
            safety_checker=None,
            requires_safety_checker=False,
            variant="fp16" if device.type == "cuda" else None
        )
        pipe.to(device)

        # تحسين الأداء
        if device.type == "cuda":
            pipe.enable_attention_slicing()
            pipe.enable_xformers_memory_efficient_attention()

        print("✅ تم تحميل النموذج الأساسي بنجاح")

        # 2️⃣ محاولة تحميل LoRA (بثلاث طرق مختلفة)
        lora_loaded = False
        if lora_path and os.path.exists(lora_path):
            print("\n🔧 2. محاولة تحميل أوزان LoRA...")

            # الطريقة 1: load_attn_procs (الأحدث في diffusers 0.26.3)
            try:
                print("   📥 الطريقة 1: استخدام load_attn_procs")
                pipe.unet.load_attn_procs(lora_path)
                print("   ✅ تم تحميل أوزان LoRA بنجاح (load_attn_procs)")
                pipe.unet.lora_loaded = True
                lora_loaded = True
            except Exception as e1:
                print(f"   ⚠️ فشل load_attn_procs: {e1}")

                # الطريقة 2: محاولة تحميل مباشر للملف
                try:
                    print("   📥 الطريقة 2: البحث عن ملف .safetensors")
                    safetensors_files = glob.glob(os.path.join(lora_path, "*.safetensors"))
                    if safetensors_files:
                        from safetensors.torch import load_file
                        lora_weights = load_file(safetensors_files[0])
                        print(f"   ✅ تم تحميل ملف {os.path.basename(safetensors_files[0])}")
                        pipe.unet.lora_loaded = True
                        lora_loaded = True
                    else:
                        print("   ⚠️ لا توجد ملفات safetensors")
                except Exception as e2:
                    print(f"   ⚠️ فشل التحميل المباشر: {e2}")
                    lora_loaded = False

            # طباعة النتيجة النهائية
            if lora_loaded:
                print("\n✅✅✅ تم تحميل LoRA بنجاح! ✅✅✅")
            else:
                print("\n⚠️⚠️⚠️ فشل تحميل LoRA، استخدام النموذج الأساسي ⚠️⚠️⚠️")
        else:
            print("\n⚠️ استخدام النموذج الأساسي بدون LoRA")

        # 3️⃣ إعداد scheduler
        print("\n🔧 3. إعداد scheduler...")
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        print("✅ تم إعداد scheduler")

        print("\n✅✅✅ تم تحميل النموذج بنجاح ✅✅✅")
        return pipe

    except Exception as e:
        print(f"\n❌❌❌ خطأ في تحميل النموذج: {e} ❌❌❌")
        import traceback
        traceback.print_exc()

        # محاولة أخيرة - تحميل بدون float16
        print("\n🔄 محاولة تحميل النموذج بدون float16...")
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            MODEL_NAME,
            safety_checker=None
        ).to(device)
        return pipe

# ===================== 1️⃣4️⃣ دوال معالجة Stable Diffusion =====================
def create_mouth_mask(pil_image: Image.Image, face_bbox):
    """إنشاء ماسك للفم"""
    img = np.array(pil_image.convert("RGB"))[:, :, ::-1].copy()
    h, w = img.shape[:2]
    x_min, y_min, x_max, y_max = face_bbox

    face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5)
    results = face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if not results.multi_face_landmarks:
        return None

    mask = np.zeros((h, w), dtype=np.uint8)
    for face_landmarks in results.multi_face_landmarks:
        inner_lip_indices = [
            78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
            308, 324, 318, 402, 317, 14, 87, 178, 88, 95
        ]
        points = []
        for idx in inner_lip_indices:
            x = int(face_landmarks.landmark[idx].x * w)
            y = int(face_landmarks.landmark[idx].y * h)
            points.append((x, y))
        points = np.array(points, dtype=np.int32)

        kernel = np.ones((5, 5), np.uint8)
        temp_mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(temp_mask, [points], 255)
        temp_mask = cv2.dilate(temp_mask, kernel, iterations=1)
        temp_mask = cv2.GaussianBlur(temp_mask, (5, 5), 0)
        mask = cv2.add(mask, temp_mask)

    mask_cropped = mask[y_min:y_max, x_min:x_max]
    return mask_cropped

def run_inpainting_on_image_advanced(pipe, pil_img, mask_img, prompt, negative_prompt=None,
                                     seed=42, num_inference_steps=50, strength=0.7, guidance_scale=7.5):
    """
    دالة متقدمة لـ Inpainting مع الإعدادات الذهبية للأسنان
    """
    init_image = pil_img.convert("RGB")
    mask_image = mask_img.convert("L")

    default_negative_prompt = (
        "yellow teeth, stained, decayed, crooked, gum disease, inflammation, deformed teeth, "
        "extra teeth, cartoon, blurry, unrealistic, overexposed, low quality, artifacts"
    )

    if negative_prompt is None or negative_prompt.strip() == "":
        negative_prompt = default_negative_prompt

    generator = torch.Generator(device=pipe.device).manual_seed(seed)

    print(f"\n🎯 الإعدادات الذهبية للتشغيل:")
    print(f"   • عدد الخطوات: {num_inference_steps}")
    print(f"   • قوة التعديل: {strength}")
    print(f"   • مقياس الإرشاد: {guidance_scale}")
    print(f"   • Seed: {seed}")
    print(f"   • Prompt: {prompt[:50]}...")
    print(f"   • Negative Prompt: {negative_prompt[:50]}...")

    output_image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        mask_image=mask_image,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        strength=strength,
        generator=generator,
        num_images_per_prompt=1,
        width=512,
        height=512
    ).images[0]

    print("✅ تم الانتهاء من المعالجة بنجاح")
    return output_image

def segment_face_only_sd(image_array, face_mesh, margin=20, output_size=512):
    """قص الوجه لنموذج SD"""
    try:
        image_bgr = image_array
        height, width, _ = image_bgr.shape
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image_rgb)

        if not results.multi_face_landmarks:
            return None, None

        face_landmarks = results.multi_face_landmarks[0]
        face_oval_indices = mp.solutions.face_mesh.FACEMESH_FACE_OVAL

        points = []
        for idx_tuple in face_oval_indices:
            idx = idx_tuple[0]
            lm = face_landmarks.landmark[idx]
            points.append([int(lm.x * width), int(lm.y * height)])

        points = np.array(points)
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)

        x_min = max(x_min - margin, 0)
        y_min = max(y_min - margin, 0)
        x_max = min(x_max + margin, width - 1)
        y_max = min(y_max + margin, height - 1)

        face_bbox = (x_min, y_min, x_max, y_max)
        face_crop = image_bgr[y_min:y_max, x_min:x_max]

        mask_crop = np.zeros_like(face_crop[:, :, 0])
        shifted_points = points - [x_min, y_min]
        hull = cv2.convexHull(shifted_points)
        cv2.fillConvexPoly(mask_crop, hull, 255)

        final_crop = np.ones_like(face_crop, dtype=np.uint8) * 255
        final_crop[mask_crop == 255] = face_crop[mask_crop == 255]

        final_resized = cv2.resize(final_crop, (output_size, output_size), interpolation=cv2.INTER_AREA)
        final_pil = Image.fromarray(cv2.cvtColor(final_resized, cv2.COLOR_BGR2RGB))

        return final_pil, face_bbox

    except Exception as e:
        print(f"✗ خطأ في قص الوجه: {e}")
        return None, None

def extract_mouth_precise_sd(image, landmarks):
    """استخراج الفم بدقة لنموذج SD"""
    mouth_indices = get_mouth_indices_mediapipe()
    mouth_points = [landmarks[i] for i in mouth_indices if i < len(landmarks)]

    if len(mouth_points) < 3:
        raise ValueError("لم يتم العثور على نقاط فم كافية")

    mouth_points_array = np.array(mouth_points, dtype=np.int32)
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    hull = cv2.convexHull(mouth_points_array)
    cv2.fillConvexPoly(mask, hull, 255)

    x, y, w, h = cv2.boundingRect(hull)
    x, y, w, h = ensure_roi_in_bounds(image, x, y, w, h)

    margin = 2
    x = max(0, x - margin)
    y = max(0, y - margin)
    w = min(image.shape[1] - x, w + 2 * margin)
    h = min(image.shape[0] - y, h + 2 * margin)
    x, y, w, h = ensure_roi_in_bounds(image, x, y, w, h)

    mouth_cropped = image[y:y+h, x:x+w].copy()
    mask_cropped = mask[y:y+h, x:x+w].copy()

    return mouth_cropped, mask_cropped, (x, y, w, h)

def align_and_replace_mouth(original_image, enhanced_face, original_landmarks, enhanced_landmarks):
    """محاذاة واستبدال الفم"""
    try:
        enhanced_mouth, enhanced_mask, enhanced_rect = extract_mouth_precise_sd(enhanced_face, enhanced_landmarks)
        original_mouth, original_mask, original_rect = extract_mouth_precise_sd(original_image, original_landmarks)

        x, y, w, h = original_rect
        resized_mouth = cv2.resize(enhanced_mouth, (w, h))
        resized_mask = cv2.resize(enhanced_mask, (w, h))

        mask_float = resized_mask.astype(np.float32) / 255.0
        result = original_image.copy()
        region_result = result[y:y+h, x:x+w].astype(np.float32)
        mouth_float = resized_mouth.astype(np.float32)

        for c in range(3):
            region_result[:, :, c] = region_result[:, :, c] * (1 - mask_float) + mouth_float[:, :, c] * mask_float

        result[y:y+h, x:x+w] = region_result.astype(np.uint8)
        return result

    except Exception as e:
        print(f"خطأ في محاذاة الفم: {e}")
        return original_image

def apply_final_enhancement(image_array, output_dir):
    """تطبيق التحسين النهائي"""
    try:
        temp_input = tempfile.NamedTemporaryFile(delete=False, suffix=".jpg")
        cv2.imwrite(temp_input.name, image_array)

        temp_output = os.path.join(output_dir, "final_enhanced.jpg")
        return_code, log = run_codeformer_restore(temp_input.name, temp_output, w=0.7)

        if os.path.exists(temp_output) and os.path.getsize(temp_output) > 0:
            enhanced_image = cv2.imread(temp_output)
            print("✅ تم تحسين الصورة باستخدام CodeFormer")
        else:
            enhance_image_alternative(temp_input.name, temp_output)
            enhanced_image = cv2.imread(temp_output)
            print("✅ تم تحسين الصورة باستخدام التحسين البديل")

        os.unlink(temp_input.name)
        return enhanced_image

    except Exception as e:
        print(f"❌ خطأ في التحسين النهائي: {e}")
        return image_array

# ===================== 1️⃣5️⃣ تحميل النماذج =====================
print("\n" + "="*60)
print("🚀 بدء تحميل جميع النماذج...")
print("="*60)

# تحميل نماذج التقييم
lpips_model, clip_model, clip_preprocess, eval_device = load_evaluation_models()

# تحميل نموذج Pix2Pix
pix2pix_model, pix2pix_transform, device = load_pix2pix_model()

# تحميل نموذج SD Inpainting مع LoRA
sd_pipe = load_sd_inpainting_model()

# تهيئة FaceMesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5)

print("\n" + "="*60)
print("✅✅✅ تم تحميل جميع النماذج بنجاح ✅✅✅")
print("="*60 + "\n")

# ===================== 1️⃣6️⃣ واجهات FastAPI =====================
@app.get("/", response_class=HTMLResponse)
async def home(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})

@app.get("/evaluate", response_class=HTMLResponse)
async def evaluate_page(request: Request):
    return templates.TemplateResponse("evaluate.html", {"request": request})

@app.post("/process/{model_type}")
async def process_image(
    model_type: str,
    file: UploadFile = File(...),
    prompt: str = Form("Dental treatment using zirconium crowns and composite fillings, high quality, clear detail, Beautiful white teeth."),
    negative_prompt: str = Form("yellow teeth, stained, decayed, crooked, gum disease, inflammation, deformed teeth, extra teeth, cartoon, blurry, unrealistic, overexposed, low quality, artifacts"),
    seed: int = Form(1234)
):
    # تنظيف والتحقق من البرومبت
    cleaned_prompt = sanitize_prompt(prompt)
    cleaned_negative_prompt = sanitize_prompt(negative_prompt) if negative_prompt else None

    if not is_valid_prompt(cleaned_prompt):
        raise HTTPException(
            status_code=400,
            detail="❌ البرومبت غير صالح. يرجى استخدام الأحرف العربية والإنجليزية فقط مع رموز أساسية"
        )

    temp_dir = None
    try:
        temp_dir = tempfile.mkdtemp()
        orig_path = os.path.join(temp_dir, "original.jpg")

        # حفظ الصورة المرفوعة
        with open(orig_path, "wb") as f:
            content = await file.read()
            f.write(content)

        if model_type == "pix2pix":
            # معالجة Pix2Pix
            image = Image.open(orig_path).convert("RGB")

            # قص الوجه
            face_img = segment_face_only(image, face_mesh)
            if face_img is None:
                return JSONResponse(content={"error": "مافي وجه بالصورة"}, status_code=400)

            # معالجة Pix2Pix
            input_tensor = pix2pix_transform(face_img).unsqueeze(0).to(device)
            with torch.no_grad():
                output_tensor = pix2pix_model(input_tensor)

            output_tensor = output_tensor.squeeze(0).cpu()
            output_tensor = (output_tensor + 1) / 2
            output_image = transforms.ToPILImage()(output_tensor)
            enhanced_path = os.path.join(temp_dir, "enhanced_face.jpg")
            output_image.save(enhanced_path, quality=95)

            # استبدال الفم
            final_result = replace_mouth_natural(enhanced_path, orig_path)
            final_before_codeformer = os.path.join(temp_dir, "final_before_codeformer.jpg")
            cv2.imwrite(final_before_codeformer, final_result, [cv2.IMWRITE_JPEG_QUALITY, 95])

            # تطبيق CodeFormer لتحسين الجودة النهائية
            final_after_codeformer = os.path.join(temp_dir, "final_after_codeformer.jpg")
            return_code, log = run_codeformer_restore(final_before_codeformer, final_after_codeformer, w=0.7)

            if os.path.exists(final_after_codeformer) and os.path.getsize(final_after_codeformer) > 0:
                final_path = final_after_codeformer
                print("✅ تم استخدام الصورة بعد CodeFormer")
            else:
                final_path = final_before_codeformer
                print("⚠️ استخدام الصورة قبل CodeFormer بسبب مشكلة في المعالجة")

        elif model_type == "sd_inpainting":
            # معالجة Stable Diffusion Inpainting
            original_img = Image.open(orig_path).convert("RGB")
            original_cv = np.array(original_img.convert("RGB"))[:, :, ::-1].copy()

            # معالجة الوجه والفم
            face_crop, face_bbox = segment_face_only_sd(original_cv, face_mesh, margin=20, output_size=512)
            if face_crop is None:
                return JSONResponse({"error": "لم يتم العثور على وجه"}, status_code=400)

            mouth_mask = create_mouth_mask(face_crop, (0, 0, 512, 512))
            if mouth_mask is None:
                return JSONResponse({"error": "لم يتم العثور على الفم"}, status_code=400)

            # استخدام الإعدادات الذهبية
            enhanced_teeth = run_inpainting_on_image_advanced(
                sd_pipe,
                face_crop,
                Image.fromarray(mouth_mask),
                prompt=cleaned_prompt,
                negative_prompt=cleaned_negative_prompt,
                seed=seed
            )

            enhanced_face_cv = np.array(enhanced_teeth)[:, :, ::-1].copy()

            # دمج الفم المحسن مع الصورة الأصلية
            original_landmarks = detect_landmarks_mediapipe(original_cv)
            enhanced_landmarks = detect_landmarks_mediapipe(enhanced_face_cv)
            merged_full_cv = align_and_replace_mouth(original_cv, enhanced_face_cv, original_landmarks, enhanced_landmarks)

            # تطبيق التحسين النهائي
            final_enhanced_cv = apply_final_enhancement(merged_full_cv, temp_dir)
            final_path = os.path.join(temp_dir, "final_sd_result.jpg")
            cv2.imwrite(final_path, final_enhanced_cv, [cv2.IMWRITE_JPEG_QUALITY, 95])

            # حفظ إعدادات التشغيل
            settings_path = os.path.join(temp_dir, "settings.txt")
            with open(settings_path, "w", encoding="utf-8") as f:
                f.write(f"""إعدادات المعالجة:
Prompt: {cleaned_prompt}
Negative Prompt: {cleaned_negative_prompt}
Seed: {seed}
عدد الخطوات: 50 (ذهبي)
قوة التعديل: 0.7 (ذهبي)
مقياس الإرشاد: 7.5 (ذهبي)
نموذج: Stable Diffusion Inpainting
تاريخ: {time.strftime('%Y-%m-%d %H:%M:%S')}
""")

        else:
            return JSONResponse(content={"error": "نوع النموذج غير معروف"}, status_code=400)

        # التحقق النهائي من وجود الملف
        if not os.path.exists(final_path):
            raise FileNotFoundError("الملف النهائي غير موجود")

        # إرجاع النتيجة
        response = FileResponse(final_path, media_type="image/jpeg",
                                filename=f"enhanced_teeth_s{seed}.jpg")

        # تنظيف الملفات المؤقتة بعد إرسال الرد
        async def cleanup():
            await asyncio.sleep(2)
            shutil.rmtree(temp_dir, ignore_errors=True)

        asyncio.create_task(cleanup())

        return response

    except Exception as e:
        # تنظيف الملفات المؤقتة في حالة الخطأ
        if temp_dir and os.path.exists(temp_dir):
            shutil.rmtree(temp_dir, ignore_errors=True)
        return JSONResponse(content={"error": f"خطأ في المعالجة: {str(e)}"}, status_code=500)

@app.post("/evaluate")
async def evaluate_images(real: UploadFile = File(...), fake: UploadFile = File(...)):
    try:
        temp_dir = tempfile.mkdtemp()
        real_path = os.path.join(temp_dir, "real_temp.png")
        fake_path = os.path.join(temp_dir, "fake_temp.png")

        with open(real_path, "wb") as f:
            f.write(await real.read())
        with open(fake_path, "wb") as f:
            f.write(await fake.read())

        lpips_val, clip_val, final_val = compute_similarity(
            real_path, fake_path, lpips_model, clip_model, clip_preprocess, eval_device
        )

        if lpips_val is None:
            return JSONResponse(content={"error": "لم يتم العثور على وجه في إحدى الصور"}, status_code=400)

        # تنظيف الملفات المؤقتة
        shutil.rmtree(temp_dir, ignore_errors=True)

        return {
            "LPIPS_similarity": f"{lpips_val:.2f}%",
            "CLIP_similarity": f"{clip_val:.2f}%",
            "Final_similarity": f"{final_val:.2f}%"
        }
    except Exception as e:
        return JSONResponse(content={"error": str(e)}, status_code=500)

@app.get("/health")
async def health_check():
    return {"status": "healthy", "timestamp": time.time(), "models_loaded": True}

@app.get("/lora-status")
async def lora_status():
    """التحقق من حالة تحميل LoRA"""
    lora_loaded = False
    if hasattr(sd_pipe, 'unet'):
        lora_loaded = hasattr(sd_pipe.unet, 'lora_loaded') and sd_pipe.unet.lora_loaded

    status = {
        "model_type": "Stable Diffusion Inpainting",
        "lora_loaded": lora_loaded,
        "lora_path_found": os.path.exists('/content/drive/MyDrive/inpaint_lora_output_ffinal')
    }
    return status

# ===================== 1️⃣7️⃣ واجهات HTML =====================
html_content = """
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>تحسين مظهر الأسنان باستخدام الذكاء الاصطناعي</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: rgb(179, 205, 215);
            text-align: center;
        }
        .container {
            max-width: 1000px;
            margin: 0 auto;
            background-color: rgb(25, 209, 230);
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0,0,0,0.1);
        }
        h1 {
            color: #052a3d;
        }
        .nav-buttons {
            display: flex;
            justify-content: center;
            gap: 20px;
            margin: 20px 0;
        }
        .nav-btn {
            padding: 15px 30px;
            font-size: 18px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            transition: all 0.3s;
            text-decoration: none;
            color: white;
            display: inline-block;
        }
        .process-btn {
            background-color: #117192;
        }
        .evaluate-btn {
            background-color: rgb(179, 205, 215);
            color: #052a3d;
        }
        .nav-btn:hover {
            opacity: 0.9;
            transform: translateY(-2px);
        }
        .model-buttons {
            display: flex;
            justify-content: center;
            gap: 20px;
            margin: 20px 0;
        }
        .model-btn {
            padding: 15px 30px;
            font-size: 18px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            transition: all 0.3s;
        }
        .pix2pix-btn {
            background-color: #117192;
            color: white;
        }
        .sd-btn {
            background-color: rgb(179, 205, 215);
            color: #052a3d;
        }
        .model-btn:hover {
            opacity: 0.9;
            transform: translateY(-2px);
        }
        .upload-section {
            margin: 20px 0;
            display: none;
        }
        .prompt-section {
            margin: 20px 0;
            display: none;
        }
        .seed-section {
            margin: 20px 0;
            display: none;
        }
        .result-section {
            display: flex;
            justify-content: space-around;
            margin-top: 30px;
        }
        .result-box {
            width: 45%;
            background-color: white;
            padding: 15px;
            border-radius: 10px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .result-box h3 {
            color: #052a3d;
            margin-top: 0;
        }
        .result-box img {
            max-width: 100%;
            border: 1px solid #ddd;
            border-radius: 5px;
            padding: 5px;
        }
        .loading {
            display: none;
            margin: 20px 0;
            color: #7f8c8d;
        }
        .process-action-btn {
            padding: 15px 30px;
            background-color: #052a3d;
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-size: 18px;
            margin-top: 20px;
            transition: all 0.3s;
        }
        .process-action-btn:hover {
            background-color: rgb(17, 113, 146);
            transform: translateY(-2px);
        }
        .process-action-btn:disabled {
            background-color: #95a5a6;
            cursor: not-allowed;
        }
        .download-btn {
            padding: 10px 20px;
            background-color: #052a3d;
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            margin-top: 15px;
            display: none;
        }
        .download-btn:hover {
            background-color: #117192;
        }
        .error-message {
            color: red;
            margin: 10px 0;
            display: none;
        }
        .valid-message {
            color: green;
            margin: 10px 0;
            display: none;
        }
        .info-box {
            background-color: #e8f4f8;
            border-right: 4px solid #117192;
            padding: 15px;
            margin: 15px 0;
            border-radius: 5px;
            text-align: right;
        }
        .info-box h4 {
            color: #052a3d;
            margin-top: 0;
        }
        #seedValue {
            padding: 8px;
            border: 1px solid #ddd;
            border-radius: 5px;
            width: 150px;
            text-align: center;
            margin-top: 5px;
        }
        .seed-info {
            font-size: 12px;
            color: #666;
            margin-top: 5px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>تحسين مظهر الأسنان باستخدام الذكاء الاصطناعي</h1>
        <p>اختر الخدمة التي تريد استخدامها</p>

        <div class="nav-buttons">
            <a href="/" class="nav-btn process-btn">معالجة الصور</a>
            <a href="/evaluate" class="nav-btn evaluate-btn">تقييم الجودة</a>
        </div>

        <div class="info-box">
            <h4>🎯 الإعدادات الذهبية المثبتة:</h4>
            <p>• عدد الخطوات: <strong>50</strong> (مثالي للجودة والسرعة)</p>
            <p>• قوة التعديل: <strong>0.7</strong> (مثالي للأسنان)</p>
            <p>• مقياس الإرشاد: <strong>7.5</strong> (واقعي وطبيعي)</p>
            <p>• يمكنك تغيير الـ Prompt والـ Seed فقط</p>
        </div>

        <h2>معالجة الصور</h2>
        <p>اختر النموذج الذي تريد استخدامه لتحسين مظهر الأسنان في صورتك</p>

        <div class="model-buttons">
            <button class="model-btn pix2pix-btn" onclick="selectModel('pix2pix')">نموذج بدون Prompt</button>
            <button class="model-btn sd-btn" onclick="selectModel('sd_inpainting')">نموذج مع Prompt</button>
        </div>

        <div id="uploadSection" class="upload-section">
            <h3>اختر صورة لمعالجتها</h3>
            <input type="file" id="imageFile" accept="image/*" onchange="previewImage()">
        </div>

        <div id="promptSection" class="prompt-section">
            <h3>أدخل وصف النتيجة المطلوبة</h3>
            <textarea id="promptText" rows="3" cols="60" oninput="validatePrompt()">Dental treatment using zirconium crowns and composite fillings, high quality, clear detail, Beautiful white teeth</textarea>
            <div id="promptError" class="error-message">
                ❌ استخدم الأحرف العربية والإنجليزية فقط مع رموز أساسية
            </div>
            <div id="promptValid" class="valid-message">
                ✅ البرومبت صالح
            </div>

            <div style="margin-top: 15px;">
                <h4>Negative Prompt (اختياري)</h4>
                <textarea id="negativePrompt" rows="2" cols="60">yellow teeth, stained, decayed, crooked, gum disease, inflammation, deformed teeth, extra teeth, cartoon, blurry, unrealistic, overexposed, low quality, artifacts</textarea>
            </div>
        </div>

        <div id="seedSection" class="seed-section">
            <h3>Seed للتكرارية</h3>
            <input type="number" id="seedValue" value="1234" min="0" max="9999999">
            <div class="seed-info">نفس الـ Seed يعطي نفس النتيجة كل مرة. قم بتغييره للحصول على نتائج مختلفة.</div>
        </div>

        <div id="loading" class="loading">
            <p>جاري معالجة الصورة... يرجى الانتظار (تستغرق حوالي 2-3 دقائق)</p>
        </div>

        <button id="processBtn" class="process-action-btn" style="display: none;" onclick="processImage()">معالجة الصورة</button>

        <div class="result-section">
            <div class="result-box">
                <h3>الصورة الأصلية</h3>
                <img id="originalImage" src="" alt="الصورة الأصلية">
            </div>
            <div class="result-box">
                <h3>النتيجة بعد المعالجة</h3>
                <img id="resultImage" src="" alt="النتيجة بعد المعالجة">
                <button id="downloadBtn" class="download-btn" onclick="downloadImage()">تنزيل الصورة</button>
            </div>
        </div>
    </div>

    <script>
        let selectedModel = '';
        let resultImageUrl = '';
        let isPromptValid = true;

        function selectModel(model) {
            selectedModel = model;
            document.getElementById('uploadSection').style.display = 'block';
            document.getElementById('processBtn').style.display = 'block';

            if (model === 'sd_inpainting') {
                document.getElementById('promptSection').style.display = 'block';
                document.getElementById('seedSection').style.display = 'block';
                // تعيين الـ prompt الذهبي تلقائيًا
                document.getElementById('promptText').value = "Dental treatment using zirconium crowns and composite fillings, high quality, clear detail, Beautiful white teeth.";
                validatePrompt();
                // توليد Seed عشوائي تلقائيًا
                document.getElementById('seedValue').value = Math.floor(Math.random() * 10000000);
            } else {
                document.getElementById('promptSection').style.display = 'none';
                document.getElementById('seedSection').style.display = 'none';
                isPromptValid = true;
            }
            updateProcessButton();
        }

        function validatePrompt() {
            const promptText = document.getElementById('promptText').value;
            const errorDiv = document.getElementById('promptError');
            const validDiv = document.getElementById('promptValid');

            // تحقق من الرموز المتتالية
            const multipleSymbols = /[,._\-!?،]{2,}/.test(promptText);
            // تحقق من الرموز في البداية أو النهاية
            const startsWithSymbol = /^[,._\-!?،]/.test(promptText);
            const endsWithSymbol = /[,._\-!?،]$/.test(promptText);
            // تحقق من الأحرف المسموحة فقط
            const validChars = /^[a-zA-Z\u0600-\u06FF\s,._\-!?،]*$/.test(promptText);

            isPromptValid = !multipleSymbols && !startsWithSymbol && !endsWithSymbol && validChars;

            if (!isPromptValid) {
                errorDiv.style.display = 'block';
                validDiv.style.display = 'none';
            } else {
                errorDiv.style.display = 'none';
                validDiv.style.display = 'block';
            }

            updateProcessButton();
        }

        function updateProcessButton() {
            const processBtn = document.getElementById('processBtn');
            const hasFile = document.getElementById('imageFile').files.length > 0;

            if (selectedModel === 'sd_inpainting') {
                processBtn.disabled = !hasFile || !isPromptValid;
            } else {
                processBtn.disabled = !hasFile;
            }
        }

        function previewImage() {
            const file = document.getElementById('imageFile').files[0];
            if (file) {
                const reader = new FileReader();
                reader.onload = function(e) {
                    document.getElementById('originalImage').src = e.target.result;
                }
                reader.readAsDataURL(file);
                updateProcessButton();
            }
        }

        async function processImage() {
            const fileInput = document.getElementById('imageFile');
            const promptText = document.getElementById('promptText').value;
            const negativePrompt = document.getElementById('negativePrompt').value;
            const seedValue = document.getElementById('seedValue').value;

            // التحقق من المدخلات
            if (!fileInput.files.length) {
                alert('يرجى اختيار صورة أولاً');
                return;
            }

            if (selectedModel === 'sd_inpainting' && !isPromptValid) {
                alert('يرجى إدخال برومبت صالح');
                return;
            }

            document.getElementById('loading').style.display = 'block';
            document.getElementById('downloadBtn').style.display = 'none';

            const formData = new FormData();
            formData.append('file', fileInput.files[0]);
            formData.append('prompt', promptText);
            formData.append('negative_prompt', negativePrompt);
            formData.append('seed', seedValue);

            try {
                const response = await fetch(`/process/${selectedModel}`, {
                    method: 'POST',
                    body: formData
                });

                if (!response.ok) {
                    const error = await response.json();
                    throw new Error(error.detail || 'حدث خطأ غير معروف');
                }

                const blob = await response.blob();
                resultImageUrl = URL.createObjectURL(blob);
                document.getElementById('resultImage').src = resultImageUrl;
                document.getElementById('downloadBtn').style.display = 'block';

                // إظهار تفاصيل الإعدادات
                alert(`✅ تم معالجة الصورة بنجاح!\\n\\nالإعدادات المستخدمة:\\n• الإعدادات الذهبية مثبتة:\\n   - عدد الخطوات: 50\\n   - قوة التعديل: 0.7\\n   - مقياس الإرشاد: 7.5\\n• Seed: ${seedValue}\\n\\nنفس الـ Seed يعطي نفس النتيجة دائماً`);

            } catch (error) {
                console.error('Error:', error);
                alert('حدث خطأ أثناء معالجة الصورة: ' + error.message);
            } finally {
                document.getElementById('loading').style.display = 'none';
            }
        }

        function downloadImage() {
            if (!resultImageUrl) {
                alert('لا توجد صورة معالجة للتنزيل');
                return;
            }

            const a = document.createElement('a');
            a.href = resultImageUrl;
            a.download = 'enhanced_teeth_image.jpg';
            document.body.appendChild(a);
            a.click();
            document.body.removeChild(a);
        }
    </script>
</body>
</html>
"""

evaluate_html_content = """
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>تقييم جودة تحسين الأسنان</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: rgb(179, 205, 215);
            text-align: center;
        }
        .container {
            max-width: 1000px;
            margin: 0 auto;
            background-color: rgb(25, 209, 230);
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0,0,0,0.1);
        }
        h1 {
            color: #052a3d;
        }
        .nav-buttons {
            display: flex;
            justify-content: center;
            gap: 20px;
            margin: 20px 0;
        }
        .nav-btn {
            padding: 15px 30px;
            font-size: 18px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            transition: all 0.3s;
            text-decoration: none;
            color: white;
            display: inline-block;
        }
        .process-btn {
            background-color: #117192;
        }
        .evaluate-btn {
            background-color: rgb(179, 205, 215);
        }
        .nav-btn:hover {
            opacity: 0.9;
            transform: translateY(-2px);
        }
        .upload-section {
            display: flex;
            justify-content: space-around;
            margin: 30px 0;
        }
        .upload-box {
            width: 45%;
            border: 2px dashed #ddd;
            padding: 20px;
            border-radius: 10px;
        }
        .upload-box h3 {
            margin-top: 0;
            color: #052a3d;
        }
        .upload-box img {
            max-width: 100%;
            max-height: 300px;
            margin-top: 15px;
            border-radius: 5px;
            border: 1px solid #ddd;
        }
        .results {
            margin: 30px 0;
            padding: 20px;
            background-color: #f9f9f9;
            border-radius: 10px;
            display: none;
        }
        .result-item {
            display: flex;
            justify-content: space-between;
            margin: 10px 0;
            padding: 15px;
            background-color: white;
            border-radius: 5px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .result-label {
            font-weight: bold;
            color: #052a3d;
        }
        .result-value {
            color: #19d1e6;
            font-weight: bold;
        }
        .loading {
            display: none;
            margin: 20px 0;
            color: #7f8c8d;
        }
        .evaluate-action-btn {
            padding: 15px 30px;
            background-color: #052a3d;
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-size: 18px;
            margin-top: 20px;
            transition: all 0.3s;
        }
        .evaluate-action-btn:hover {
            background-color: rgb(17, 113, 146);
            transform: translateY(-2px);
        }
        .evaluate-action-btn:disabled {
            background-color: #95a5a6;
            cursor: not-allowed;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>تقييم جودة تحسين الأسنان</h1>
        <p>قم برفع صورتين (أصلية ومعالجة) لتقييم جودة التحسين باستخدام نماذج الذكاء الاصطناعي</p>

        <div class="nav-buttons">
            <a href="/" class="nav-btn process-btn">معالجة الصور</a>
            <a href="/evaluate" class="nav-btn evaluate-btn">تقييم الجودة</a>
        </div>

        <div class="upload-section">
            <div class="upload-box">
                <h3>الصورة الأصلية</h3>
                <input type="file" id="realImage" accept="image/*" onchange="previewRealImage()">
                <img id="realPreview" src="" alt="معاينة الصورة الأصلية">
            </div>
            <div class="upload-box">
                <h3>الصورة المعالجة</h3>
                <input type="file" id="fakeImage" accept="image/*" onchange="previewFakeImage()">
                <img id="fakePreview" src="" alt="معاينة الصورة المعالجة">
            </div>
        </div>

        <div id="loading" class="loading">
            <p>جاري تقييم الجودة... يرجى الانتظار</p>
        </div>

        <button id="evaluateBtn" class="evaluate-action-btn" onclick="evaluateImages()">تقييم الجودة</button>

        <div id="results" class="results">
            <h2>نتائج التقييم</h2>
            <div class="result-item">
                <span class="result-label">تشابه LPIPS:</span>
                <span id="lpipsResult" class="result-value">-</span>
            </div>
            <div class="result-item">
                <span class="result-label">تشابه CLIP:</span>
                <span id="clipResult" class="result-value">-</span>
            </div>
            <div class="result-item">
                <span class="result-label">التقييم النهائي:</span>
                <span id="finalResult" class="result-value">-</span>
            </div>
        </div>
    </div>

    <script>
        function previewRealImage() {
            const file = document.getElementById('realImage').files[0];
            const preview = document.getElementById('realPreview');
            if (file) {
                const reader = new FileReader();
                reader.onload = function(e) {
                    preview.src = e.target.result;
                    preview.style.display = 'block';
                    checkEvaluateButton();
                }
                reader.readAsDataURL(file);
            } else {
                preview.style.display = 'none';
                checkEvaluateButton();
            }
        }

        function previewFakeImage() {
            const file = document.getElementById('fakeImage').files[0];
            const preview = document.getElementById('fakePreview');
            if (file) {
                const reader = new FileReader();
                reader.onload = function(e) {
                    preview.src = e.target.result;
                    preview.style.display = 'block';
                    checkEvaluateButton();
                }
                reader.readAsDataURL(file);
            } else {
                preview.style.display = 'none';
                checkEvaluateButton();
            }
        }

        function checkEvaluateButton() {
            const realFile = document.getElementById('realImage').files[0];
            const fakeFile = document.getElementById('fakeImage').files[0];
            const evaluateBtn = document.getElementById('evaluateBtn');

            evaluateBtn.disabled = !(realFile && fakeFile);
        }

        async function evaluateImages() {
            const realFile = document.getElementById('realImage').files[0];
            const fakeFile = document.getElementById('fakeImage').files[0];

            if (!realFile || !fakeFile) {
                alert('يرجى اختيار كل من الصورة الأصلية والصورة المعالجة');
                return;
            }

            document.getElementById('loading').style.display = 'block';
            document.getElementById('results').style.display = 'none';
            document.getElementById('evaluateBtn').disabled = true;

            const formData = new FormData();
            formData.append('real', realFile);
            formData.append('fake', fakeFile);

            try {
                const response = await fetch('/evaluate', {
                    method: 'POST',
                    body: formData
                });

                if (!response.ok) {
                    const error = await response.text();
                    throw new Error(error);
                }

                const results = await response.json();

                document.getElementById('lpipsResult').textContent = results.LPIPS_similarity;
                document.getElementById('clipResult').textContent = results.CLIP_similarity;
                document.getElementById('finalResult').textContent = results.Final_similarity;

                document.getElementById('results').style.display = 'block';

            } catch (error) {
                console.error('Error:', error);
                alert('حدث خطأ أثناء تقييم الجودة: ' + error.message);
            } finally {
                document.getElementById('loading').style.display = 'none';
                document.getElementById('evaluateBtn').disabled = false;
            }
        }

        // تهيئة الصفحة
        document.addEventListener('DOMContentLoaded', function() {
            checkEvaluateButton();
        });
    </script>
</body>
</html>
"""

# حفظ واجهات HTML
with open("index.html", "w", encoding="utf-8") as f:
    f.write(html_content)

with open("evaluate.html", "w", encoding="utf-8") as f:
    f.write(evaluate_html_content)

# ===================== 1️⃣8️⃣ تشغيل الخادم =====================
def start_ngrok():
    public_url = ngrok.connect(8000, "http")
    print(f"\n✅ رابط API الجاهز عبر ngrok: {public_url}\n")
    return public_url

def signal_handler(sig, frame):
    print("\n🛑 إغلاق الخادم...")
    sys.exit(0)

if __name__ == "__main__":
    # تسجيل معالج الإشارات
    signal.signal(signal.SIGINT, signal_handler)
    signal.signal(signal.SIGTERM, signal_handler)

    public_url = start_ngrok()

    print("🚀 بدء تشغيل الخادم...")
    print(f"📊 يمكنك الوصول إلى API عبر: {public_url}")
    print("🎯 تم تثبيت الإعدادات الذهبية:")
    print("   • عدد الخطوات: 50")
    print("   • قوة التعديل: 0.7")
    print("   • مقياس الإرشاد: 7.5")
    print("   • Seed: 1234 (قابل للتغيير)")
    print("⏹️  اضغط Ctrl+C لإيقاف الخادم")

    try:
        config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
        server = uvicorn.Server(config)
        asyncio.run(server.serve())
    except KeyboardInterrupt:
        print("\n🛑 تم إيقاف الخادم بواسطة المستخدم")
    except Exception as e:
        print(f"❌ خطأ في تشغيل الخادم: {e}")
    finally:
        print("✅ تم تنظيف الموارد")

<>:1287: SyntaxWarning: invalid escape sequence '\-'
<>:1287: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipython-input-82186728.py:1287: SyntaxWarning: invalid escape sequence '\-'
  const multipleSymbols = /[,._\-!?،]{2,}/.test(promptText);
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



🚀 بدء تحميل جميع النماذج...
⏳ جارٍ تحميل نماذج التقييم (LPIPS و CLIP)...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
✅ تم تحميل نماذج التقييم بنجاح
⏳ جارٍ تحميل نموذج Pix2Pix...
🔧 جارٍ تحميل نموذج Stable Diffusion Inpainting...
📁 تم العثور على مسار LoRA: /content/drive/MyDrive/inpaint_lora_output_ffinal/checkpoint-10200
📄 محتويات المجلد:
   - adapter_model.safetensors
   - adapter_config.json
💻 استخدام الجهاز: cuda

🔧 1. تحميل النموذج الأساسي...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-118' coro=<Server.serve() done, defined at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77> exception=SystemExit(0)>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-3205206999.py", line 1705, in <cell line: 0>
    asyncio.run(server.serve())
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callba

✅ تم تحميل النموذج الأساسي بنجاح

🔧 2. محاولة تحميل أوزان LoRA...
   📥 الطريقة 1: استخدام load_attn_procs
   ⚠️ فشل load_attn_procs: Error no file named pytorch_lora_weights.bin found in directory /content/drive/MyDrive/inpaint_lora_output_ffinal/checkpoint-10200.
   📥 الطريقة 2: البحث عن ملف .safetensors
   ✅ تم تحميل ملف adapter_model.safetensors

✅✅✅ تم تحميل LoRA بنجاح! ✅✅✅

🔧 3. إعداد scheduler...
✅ تم إعداد scheduler

✅✅✅ تم تحميل النموذج بنجاح ✅✅✅

✅✅✅ تم تحميل جميع النماذج بنجاح ✅✅✅


✅ رابط API الجاهز عبر ngrok: NgrokTunnel: "https://noblemanly-subdermic-zenia.ngrok-free.dev" -> "http://localhost:8000"

🚀 بدء تشغيل الخادم...
📊 يمكنك الوصول إلى API عبر: NgrokTunnel: "https://noblemanly-subdermic-zenia.ngrok-free.dev" -> "http://localhost:8000"
🎯 تم تثبيت الإعدادات الذهبية:
   • عدد الخطوات: 50
   • قوة التعديل: 0.7
   • مقياس الإرشاد: 7.5
   • Seed: 1234 (قابل للتغيير)
⏹️  اضغط Ctrl+C لإيقاف الخادم


...

✅ تم الانتهاء من المعالجة بنجاح
✅ تم العثور على ملف الإخراج: /tmp/tmp59rq5ar2/final_results/tmpc8l81hy7.png
✅ تم تحسين الصورة باستخدام CodeFormer
INFO:     169.150.218.56:0 - "POST /process/sd_inpainting HTTP/1.1" 200 OK
✅ تم العثور على ملف الإخراج: /tmp/tmpem3u0osv/final_results/final_before_codeformer.png
✅ تم استخدام الصورة بعد CodeFormer
INFO:     129.224.207.191:0 - "POST /process/pix2pix HTTP/1.1" 200 OK
✅ تم العثور على ملف الإخراج: /tmp/tmpbrbovgc8/final_results/final_before_codeformer.png
✅ تم استخدام الصورة بعد CodeFormer
INFO:     129.224.207.191:0 - "POST /process/pix2pix HTTP/1.1" 200 OK
✅ تم العثور على ملف الإخراج: /tmp/tmpcb39slie/final_results/final_before_codeformer.png
✅ تم استخدام الصورة بعد CodeFormer
INFO:     129.224.207.191:0 - "POST /process/pix2pix HTTP/1.1" 200 OK

🎯 الإعدادات الذهبية للتشغيل:
   • عدد الخطوات: 50
   • قوة التعديل: 0.7
   • مقياس الإرشاد: 7.5
   • Seed: 1234
   • Prompt: Dental treatment using zirconium crowns and compos...
   • Negative Prompt

  0%|          | 0/35 [00:00<?, ?it/s]

✅ تم الانتهاء من المعالجة بنجاح
✅ تم العثور على ملف الإخراج: /tmp/tmpzf68cyit/final_results/tmp72goob4c.png
✅ تم تحسين الصورة باستخدام CodeFormer
INFO:     129.224.207.191:0 - "POST /process/sd_inpainting HTTP/1.1" 200 OK

🎯 الإعدادات الذهبية للتشغيل:
   • عدد الخطوات: 50
   • قوة التعديل: 0.7
   • مقياس الإرشاد: 7.5
   • Seed: 1234
   • Prompt: Dental treatment using zirconium crowns and compos...
   • Negative Prompt: yellow teeth, stained, decayed, crooked, gum disea...


  0%|          | 0/35 [00:00<?, ?it/s]

✅ تم الانتهاء من المعالجة بنجاح
✅ تم العثور على ملف الإخراج: /tmp/tmpjsae1ofr/final_results/tmpiooge56p.png
✅ تم تحسين الصورة باستخدام CodeFormer
INFO:     129.224.207.191:0 - "POST /process/sd_inpainting HTTP/1.1" 200 OK
✅ تم العثور على ملف الإخراج: /tmp/tmp67oz2ui7/final_results/final_before_codeformer.png
✅ تم استخدام الصورة بعد CodeFormer
INFO:     129.224.207.191:0 - "POST /process/pix2pix HTTP/1.1" 200 OK
